# 후속 질문 4가지 정리 — wagon 사이클 · 신호 겹침 · 밸브 딜레이 · heat/cool

이 노트북은 아래 4개 질문에 실측으로 답한다. 전부 이 репo 의 실제 함수/데이터를
그대로 불러와서 확인했다 (숫자를 다시 쓰지 않는다).

| # | 질문 | 결론 한 줄 |
|---|---|---|
| 1 | wagon 번호로 사이클을 끊어서 학습한 건 어떻게 됐고 왜 안 되나? | **모델을 만든 적이 없다** — 시각화 도구였다. 실측하면 wagon 주기 자체가 사이클이 아니라서(길이 변동계수 4.93) 애초에 학습 단위로 못 쓴다 |
| 2 | "신호 전 기울기보다 유의미하게 달라지면 exchanger 작동"이 왜 안 되나? | heat 는 안 겹친다(자체 조사 중 버그 발견·수정). **진짜 범인은 feed** — I2 는 이벤트의 63%가 feed 와 겹친다 |
| 3 | exchanger 밸브의 중복값·에러 딜레이 상관관계 | **애초에 그 태그가 없다.** feeding 에는 있는 OP/CL 딜레이 태그가 exchanger 엔 없고, 있는 유일한 태그(`Err__TT_EX`)는 5.6일간 0 만 찍힌 죽은 플래그다 |
| 4 | heat/cool 센서가 왜 안 되나? | **시도된 적이 없다.** 이유는 구조적 — heat=1-cool 완전 상보(둘 중 하나만 존재), 그리고 켜짐 비율이 tk마다 0.8~99.4%로 한쪽 상태 표본이 거의 없다 |


## 0. 설정

In [ ]:
import glob
import os
import sys

import numpy as np
import pandas as pd
from scipy import stats

sys.path.insert(0, os.path.dirname(os.getcwd()))
sys.path.insert(0, os.getcwd())

import matplotlib
import matplotlib.pyplot as plt
%matplotlib inline
matplotlib.rcParams["font.family"] = "Malgun Gothic"
matplotlib.rcParams["axes.unicode_minus"] = False

from Tank_Temp_Factor_Analysis import TANKS, WAGON_TAG
import Exchanger_Slope_Validation as EX
import Feeding_Model as FM

SAVE_DIR = os.path.join(os.path.dirname(os.getcwd()), "extracted_csv")
pd.set_option("display.width", 200)


---
# 질문 1 — wagon 사이클로 끊어서 학습한 건 어떻게 됐나

## 있었던 것 — `cycle_analysis.ipynb`

`Now_Actual_Wagon_Num`(1→42 증가 후 1로 리셋)이 줄어드는 순간을 사이클 경계로 삼아
탱크별 온도 + heat/cool/exchanger/feeding 을 사이클 단위로 잘라보는 노트북이 있다.

**그런데 이건 모델이 아니다.** `cycle_table()`(사이클 목록표), `plot_cycle()`(사이클
하나 그리기), `overlay_cycles()`(여러 사이클을 wagon 번호 기준으로 겹쳐 그리기) —
전부 **눈으로 보는 탐색 도구**다. 학습(fit)도, 예측도, 성공/실패 판정도 없다.
그래서 "왜 안 됐다"고 결론 낸 적이 없다 — 애초에 결론을 낼 단계까지 간 적이 없다.

## 그런데 왜 그 이상 진행하지 않았나 — 실측으로 확인한다

wagon 번호가 "사이클"이 되려면 그 주기가 어느 정도 일정해야 하고, 그 안에서
heat/cool/exchanger 가 "한 사이클에 한 번" 정도로 움직여야 겹쳐 볼 의미가 있다.
둘 다 사실인지 실측한다.

In [ ]:
rows = []
for tk, cfg in TANKS.items():
    p = sorted(glob.glob(os.path.join(SAVE_DIR, f"{tk}_Temp_vs_Factors_*.csv")))[-1]
    d = pd.read_csv(p, index_col="Time", parse_dates=True,
                    usecols=["Time", WAGON_TAG, cfg["heat"], cfg["exchanger"], cfg["feeding"]]).sort_index()
    d["cycle"] = (d[WAGON_TAG].diff() < 0).cumsum()
    g = d.groupby("cycle")
    dur_min = g.apply(lambda x: (x.index[-1] - x.index[0]).total_seconds() / 60)
    n_heat = g[cfg["heat"]].apply(lambda s: int((s.diff() != 0).sum()))
    n_exch = g[cfg["exchanger"]].apply(lambda s: int((s.diff() != 0).sum()))
    rows.append({"탱크": tk, "사이클수": d["cycle"].nunique(),
                 "길이_중앙(분)": round(float(dur_min.median()), 1),
                 "길이_최소~최대(분)": f"{dur_min.min():.1f}~{dur_min.max():.1f}",
                 "길이_변동계수": round(float(dur_min.std() / dur_min.mean()), 2),
                 "사이클당_heat토글_중앙": round(float(n_heat.median()), 1),
                 "사이클당_exch토글_중앙": round(float(n_exch.median()), 1)})
cycle_stats = pd.DataFrame(rows)
print("=== wagon 사이클 길이와, 그 안에서 heat/exchanger 가 몇 번 토글되나 ===")
print(cycle_stats.to_string(index=False))

## 결과 해석

**wagon 주기 자체가 불안정하다.** 중앙값은 9.1분이지만 최소 4.6분, 최대 1694분
(28시간 — 라인이 서 있던 구간이 통째로 한 사이클로 잡힌 것)까지 벌어진다.
**변동계수 4.93** — 표준편차가 평균의 5배에 가깝다는 뜻으로, "일정한 주기의 사이클"
이라 부를 수 없는 수준이다. wagon 번호는 **컨베이어 위 대차의 기계적 위치 카운터**이지
열 제어 사이클이 아니다.

**exchanger 는 한 wagon 사이클 안에서 최대 35번(P2) 토글된다.** 즉 "사이클 1개 =
exchanger 이벤트 1개"라는 대응 자체가 성립하지 않는다 — exchanger 는 wagon 과
완전히 다른, 훨씬 빠른 자기 시계로 움직인다. heat 는 사이클당 중앙값 1회로 그나마
가깝지만, 사이클 길이가 5배씩 요동치는 상황에서는 그마저 안정적인 대응이 아니다.

**결론**: wagon 사이클은 생산 공정(제품 하나 만드는 단위)의 지표이지, 온도 제어
루프의 지표가 아니다. 두 시계가 독립적으로 돌아가므로, wagon 번호로 잘라서
"이 구간의 heat/exchanger 패턴을 학습"하는 접근은 **개념 단계에서부터 안 맞는다** —
실제로 시도해서 실패한 게 아니라, 위 숫자를 보면 시도할 이유가 없다.

---
# 질문 2 — "신호 전보다 기울기가 유의미하게 다르면 exchanger 작동" 이 왜 안 되나

## 먼저 정정 — 처음 계산엔 버그가 있었다

"exchanger 이벤트 창 안에 heat 토글이 있었는가"를 처음 계산했을 때 **8~712개
전부(100%)에서 겹친다**는 결과가 나왔다. 너무 깔끔해서 의심하고 원인을 봤더니
—

```python
win = d.loc[t0-...: t0+...]       # 전체 시계열을 이 구간만큼 슬라이스
win["heat"].diff() != 0           # 슬라이스의 '첫 행'은 이전 값이 없어 diff()가 NaN
                                   # NaN != 0 은 파이썬/판다스에서 True!
```

**슬라이스 첫 행에서 매번 가짜 토글이 하나씩 잡히는 버그였다.** 전체 시계열에서
미리 토글 여부를 계산해 두고, 그 결과만 구간으로 잘라 쓰도록 고쳐서 다시 쟀다.

In [ ]:
def load_ov(tk):
    cfg = TANKS[tk]
    p = sorted(glob.glob(os.path.join(SAVE_DIR, f"{tk}_Temp_vs_Factors_*.csv")))[-1]
    raw = pd.read_csv(p, index_col="Time", parse_dates=True,
                      usecols=["Time", "source", WAGON_TAG, cfg["temp"], cfg["exchanger"],
                               cfg["heat"], cfg["feeding"]]).sort_index()
    d = pd.DataFrame({"T": raw[cfg["temp"]], "exch": raw[cfg["exchanger"]],
                      "heat": raw[cfg["heat"]], "feed": raw[cfg["feeding"]],
                      "wagon": raw[WAGON_TAG], "diag": (raw["source"] == "DIAGNOSTIC").astype(float)})
    w = d["wagon"].resample("1min").last().ffill()
    run = (w.diff().abs() > 0).rolling(30, min_periods=1).max()
    d["running"] = run.reindex(d.index, method="ffill").fillna(0)
    h = d.index.hour
    d["ok"] = (((h < 0) | (h >= 7)) & (d["diag"] == 0) & (d["running"] == 1))
    # ★ 전체 시계열에서 먼저 계산 — 슬라이스 후 diff() 금지 (첫 행 NaN 버그)
    d["heat_tog"] = d["heat"].diff().fillna(0) != 0
    return d


def cohend(a, b):
    a, b = np.asarray(a), np.asarray(b)
    n1, n2 = len(a), len(b)
    sp = np.sqrt(((n1 - 1) * a.var(ddof=1) + (n2 - 1) * b.var(ddof=1)) / (n1 + n2 - 2))
    return (a.mean() - b.mean()) / sp if sp > 0 else np.nan


def overlap_pct(d_eff):
    return float(2 * stats.norm.cdf(-abs(d_eff) / 2) * 100)


overlap_rows, dist_data = [], {}
for tk in ["P1", "I2", "I3"]:
    d = load_ov(tk)
    pre, post = EX.cycle_window(d)
    s = (d["exch"] == 1) & d["ok"]
    grp = (s & ~s.shift(1, fill_value=False)).cumsum()[s]
    ev = [(idx.iloc[0], idx.iloc[-1]) for _, idx in d.index[s].to_series().groupby(grp)
          if (idx.iloc[-1] - idx.iloc[0]).total_seconds() >= EX.MIN_EVENT_SEC]

    rows = []
    for t0, t1 in ev:
        base_seg = d.loc[t0 - pd.Timedelta(seconds=pre + post): t0 - pd.Timedelta(seconds=post), "T"].dropna()
        resp_seg = d.loc[t0: t0 + pd.Timedelta(seconds=post), "T"].dropna()
        bs, rs = EX._slope(base_seg), EX._slope(resp_seg)
        if np.isnan(bs) or np.isnan(rs):
            continue
        win = d.loc[t0 - pd.Timedelta(seconds=pre + post): t0 + pd.Timedelta(seconds=post)]
        rows.append({"dslope": rs - bs, "heat_toggled": bool(win["heat_tog"].any()),
                     "feed_on": bool((win["feed"] == 1).any())})
    e = pd.DataFrame(rows)

    clean = e[(~e["heat_toggled"]) & (~e["feed_on"])]["dslope"]
    mixed_feed = e[e["feed_on"]]["dslope"]
    dist_data[tk] = (clean, mixed_feed)

    d_feed = cohend(clean, mixed_feed) if len(clean) > 5 and len(mixed_feed) > 5 else np.nan
    overlap_rows.append({
        "탱크": tk, "전체이벤트": len(e),
        "heat도_겹침%": round(float(e["heat_toggled"].mean() * 100), 1),
        "feed도_겹침%": round(float(e["feed_on"].mean() * 100), 1),
        "완전깨끗함%": round(float(len(clean) / len(e) * 100), 1),
        "깨끗함_SNR": round(float(abs(clean.mean()) / clean.std()), 2) if len(clean) > 5 else None,
        "깨끗함vs feed겹침_Cohend": round(float(d_feed), 3) if not np.isnan(d_feed) else None,
        "분포겹침%": round(overlap_pct(d_feed), 0) if not np.isnan(d_feed) else None,
    })

print("=== (버그 수정 후) exchanger 이벤트가 다른 신호와 겹치는 정도 ===")
print(pd.DataFrame(overlap_rows).to_string(index=False))

## 결과 해석 — 범인은 heat 가 아니라 **feed**

**heat 토글은 거의 안 겹친다** (0~14%). heat 는 자기 주기가 exchanger 이벤트
창(240초)보다 훨씬 길어서, 우연히 같은 시각에 걸리는 경우가 드물다.

**feed 는 다르다** — P1 28.7%, I3 51.3%, **I2 는 63.2%**. feeding 은 원료가
탱크에 새로 들어오는 사건이라 탱크 전체의 열용량과 온도 분포를 직접 흔든다.
그 순간과 exchanger 이벤트가 겹치면 "온도 기울기가 바뀐 게 exchanger 때문인지
방금 들어온 원료 때문인지" 구분이 안 된다.

아래 히스토그램으로 직접 본다 — "깨끗함(exchanger 만 있었던 경우)"과
"feed 도 같이 켜졌던 경우"의 Δ기울기 분포가 얼마나 겹치는지.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2), sharey=False)
for ax, tk in zip(axes, ["P1", "I2", "I3"]):
    clean, mixed = dist_data[tk]
    bins = np.linspace(min(clean.min(), mixed.min()), max(clean.max(), mixed.max()), 30)
    ax.hist(clean, bins=bins, alpha=0.6, label=f"깨끗함 (n={len(clean)})", color="#2ca02c", density=True)
    ax.hist(mixed, bins=bins, alpha=0.6, label=f"feed 겹침 (n={len(mixed)})", color="#9467bd", density=True)
    ax.axvline(0, color="gray", linewidth=0.8, linestyle="--")
    ax.set_title(f"{tk} — Δ기울기 분포 비교")
    ax.set_xlabel("Δ기울기 (반응−기준, ℃/분)")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
fig.tight_layout()

## 그런데 — feed 를 걷어낸 "완전 깨끗함" 도 신호가 약하다

P1 은 완전 깨끗한 이벤트가 70.8%나 되는데도 **신호대잡음비(SNR)가 0.29** 다.
즉 **범인이 feed 만은 아니다** — 다른 신호와 안 겹쳐도, exchanger 자체의
이벤트 간 편차(같은 조건에서도 기울기가 들쭉날쭉함)가 효과 크기보다 크다.
I3(SNR 1.06)이 그나마 낫지만 이미 "이유 8"(가짜고장 주입)에서 이 정도 SNR로도
CUSUM 이 임계값을 못 넘는다는 게 확인된 바 있다 — SNR 이 볼만해도 실전 검출은
별개 문제라는 뜻이다.

---
# 질문 3 — exchanger 밸브의 중복값 · 에러 딜레이 상관관계

feeding 쪽에는 `TK_Feed_VV_OP_Err_Delay_*` / `_CL_Err_Delay_*` (밸브가 열림/닫힘
명령을 받고도 그 상태로 넘어가기까지 걸린 지연) 태그가 있어서 CL 딜레이 조기경보에
쓰고 있다. **exchanger 에도 같은 게 있는지, 있으면 상관관계를 볼 수 있는지 DB
태그 전체를 뒤져서 확인한다.**

In [ ]:
from Log_Extractor import LogExtractor
import warnings
from influxdb_client.client.warnings import MissingPivotFunction
warnings.simplefilter("ignore", MissingPivotFunction)

ROOT_DIR = os.path.dirname(os.path.dirname(os.getcwd()))  # code_ongoing -> parkkt -> 저장소 루트
ext = LogExtractor(env_path=os.path.join(ROOT_DIR, ".env"))
q = f'''
import "influxdata/influxdb/schema"
schema.tagValues(bucket: "{ext.bucket}", tag: "tag_name", predicate: (r) => r._measurement == "plc_line2", start: -7d)
'''
tagdf = ext.query_api.query_data_frame(q)
if isinstance(tagdf, list):
    tagdf = pd.concat(tagdf)
all_tags = sorted(set(tagdf["_value"].tolist()))

feed_delay = [t for t in all_tags if "Feed_VV" in t and ("Err_Delay" in t or "Off_Delay" in t)]
ex_delay = [t for t in all_tags if ("Ex" in t or "EX" in t) and ("Delay" in t or "Err" in t)]

print(f"feeding 밸브 딜레이/에러 태그: {len(feed_delay)}개 (예시)")
for t in feed_delay[:6]:
    print("  ", t)
print(f"\nexchanger 관련 Delay/Err 태그: {len(ex_delay)}개 (전체)")
for t in ex_delay:
    print("  ", t)

## 결과 — feeding 과 같은 종류의 태그가 exchanger 엔 아예 없다

`TK_Feed_VV_OP/CL_Err_Delay_*`, `STK_Feed_VV_*`, `MTK_Feed_VV_*` 처럼 **"명령 후
지연"을 재는 태그는 feeding 밸브에만 존재**한다. exchanger 에서 "Ex/EX" +
"Delay 또는 Err" 로 걸리는 건 `Err__TT_EX_*` 하나뿐인데, 이건 밸브 지연이 아니라
**온도 프로브(TT) 자체의 에러 플래그**다 — 그리고 그 프로브는 이미 알다시피
`Scale_Out___TT_EX_*` 가 전부 0인 미사용 채널이다. 그 에러 플래그가 실제로도
죽어있는지 확인한다.

In [ ]:
err_path = sorted(glob.glob(os.path.join(SAVE_DIR, "Exchanger_Err_Probe_*.csv")))[-1]
err_df = pd.read_csv(err_path, index_col=0, parse_dates=True)
err_cols = [c for c in err_df.columns if c.startswith("Err__TT_EX_")]
print(f"{os.path.basename(err_path)} — {err_df.index.min()} ~ {err_df.index.max()} ({len(err_df):,}행)\n")
for c in err_cols:
    v = err_df[c].dropna()
    print(f"  {c:<16} 고유값={sorted(v.unique())}  전부0?  {bool((v == 0).all())}")

## 결론

**"exchanger 밸브의 중복값과 에러 딜레이 값의 상관관계"는 계산할 수 있는 자료
자체가 없다.** 상관관계 분석이 실패한 게 아니라, **비교할 두 번째 변수(딜레이
값)가 이 설비에 계측돼 있지 않다.** 유일하게 이름이 걸리는 `Err__TT_EX_*` 는
5.6일 내내 0 만 찍힌 죽은 플래그이고, 그나마도 밸브가 아니라 온도 센서용이다.

**exchanger 밸브 진단에 쓸 수 있는 딜레이/피드백성 신호가 이 라인엔 전혀 없다**는
뜻이고, 이건 지금까지 나온 결론(전용 계측 부재)과 정확히 같은 방향이다 — 새로운
문제가 아니라 같은 문제가 여기서도 또 확인된 것이다.

---
# 질문 4 — heat / cool 센서가 왜 안 되나 (핵심만)

**먼저 바로잡을 것**: heat/cool 자체를 액추에이터로 진단하는 시도는 이 저장소
어디에도 없다. feeding·exchanger 만 시도됐다. 그래서 "왜 안 됐나"가 아니라
**"시도한다면 왜 막히는가"** 를 구조적으로, 그리고 heat/cool 을 exchanger 의
설명변수로 썼을 때 실측된 수치로 답한다.

In [ ]:
rows = []
for tk, cfg in TANKS.items():
    p = sorted(glob.glob(os.path.join(SAVE_DIR, f"{tk}_Temp_vs_Factors_*.csv")))[-1]
    raw = pd.read_csv(p, index_col="Time", parse_dates=True,
                      usecols=["Time", cfg["temp"], cfg["heat"], cfg["cool"]]).sort_index()
    heat, cool = raw[cfg["heat"]], raw[cfg["cool"]]
    both_off = (heat == 0) & (cool == 0)
    both_on = (heat == 1) & (cool == 1)
    n = len(raw)
    rows.append({"탱크": tk, "heat_ON%": round(float(heat.mean() * 100), 1),
                 "cool_ON%": round(float(cool.mean() * 100), 1),
                 "둘다OFF_비율%": round(float(both_off.mean() * 100), 4),
                 "둘다ON_비율%": round(float(both_on.mean() * 100), 4),
                 "둘다OFF_건수": int(both_off.sum())})
print("=== heat/cool 상호배타성 + 켜짐 비율 (8개 탱크 전수) ===")
print(pd.DataFrame(rows).to_string(index=False))
print()
print("'둘다OFF' 가 정확히 0이 아닌 탱크가 있다 — 실제로 얼마나 되는 사건인지 하나 확인한다.")
p1 = sorted(glob.glob(os.path.join(SAVE_DIR, "P1_Temp_vs_Factors_*.csv")))[-1]
r1 = pd.read_csv(p1, index_col="Time", parse_dates=True,
                 usecols=["Time", TANKS["P1"]["heat"], TANKS["P1"]["cool"]])
bo = (r1[TANKS["P1"]["heat"]] == 0) & (r1[TANKS["P1"]["cool"]] == 0)
grp = (bo != bo.shift()).cumsum()[bo]
durs = r1.index.to_series()[bo].groupby(grp).agg(lambda s: (s.iloc[-1] - s.iloc[0]).total_seconds())
print(f"[P1] 둘다OFF 클러스터 {len(durs)}개, 지속시간(초): {durs.tolist()}  "
      f"— 전체 {len(r1):,}행 중 {int(bo.sum())}행({bo.mean()*100:.4f}%)")

## 이유 ① 구조적으로 두 신호가 사실상 하나다 — `heat = 1 - cool`

"둘다 ON"은 8개 탱크 전부 0건이다. "둘다 OFF"는 **8개 탱크 전부에서 비슷하게
약 0.007%(수십 건, 각 5~40초짜리 클러스터)** 나온다 — 릴레이가 heat→cool
(또는 그 반대)로 넘어가는 찰나의 데드타임으로 보인다. 즉 **수학적으로 완벽한 항등식은 아니지만, 사실상
(>99.99%) 상보 관계**다. 회귀분석 관점에서는 이 정도 예외는 무시할 수준이라
결론은 그대로다 — 둘 다 넣으면 사실상 다중공선성이라 계수가 불안정해진다
(`ARX_Temp_Model.py` 가 heat 하나만 넣는 이유가 이것이다).

**진단 관점에서 더 중요한 건 이거다**: 신호가 하나뿐이므로 "heat 릴레이가
죽었다"와 "cool 릴레이가 죽었다"를 **구분할 방법이 원천적으로 없다.** 둘 다
같은 이진 신호의 반대쪽 얼굴이라, 이상이 생겨도 "그 신호가 이상해졌다"고만
말할 수 있지 어느 쪽 릴레이인지는 신호 자체로는 못 가른다.

## 이유 ② 켜짐 비율이 극단적으로 치우쳐 있다

I2 는 heat 가 **0.8%**밖에 안 켜진다 (사실상 항상 cool). 반대로 P3 는 cool 이
**0.6%**밖에 안 켜진다. feeding 모델에서 P4(6건)·I2(38건)가 "표본부족"으로
분류된 것과 **같은 문제**가 여기도 있다 — 소수 쪽 상태의 표본이 너무 적어서
그 상태에 대한 baseline 자체를 못 만든다.

## 이유 ③ 그런데 효과 자체는 아주 강하다 — 그게 오히려 다른 문제를 만든다

`Correlation_Analysis.py` 로 이미 측정된 값(60초 ΔT 기준, 5초 격자)을 보면
heat 의 Cohen's d 는 **0.87~2.30** 으로, exchanger(0.42~1.65의 절대값) 나
feed 보다도 크거나 비슷한 탱크가 많다. **heat/cool 은 온도를 설명하는 가장
강력한 변수다** — 이건 당연하다,애초에 온도를 제어하는 주체이기 때문이다.

**그런데 이 강함이 "이유 7"(exchanger 총정리 노트북)의 원인이다.** SHAP 로
여러 액추에이터의 기여도를 분해하면 heat/cool(=온도 상태값)이 8개 중 7개
탱크에서 기여도의 80~100%를 독점해서, exchanger 처럼 약한 신호는 그 안에
묻혀버린다. **heat/cool 은 "안 되는" 게 아니라 "너무 잘 돼서 다른 신호를
가리는" 쪽에 가깝다.**

## 이유 ④ 그래도 릴레이 자체의 건강 진단은 안 된다 — 계측 부재는 여기도 같다

exchanger 와 마찬가지로 히터의 실제 발열량(전류/전력)이나 냉각수 유량을 재는
센서가 없다. "명령이 나갔는데 온도가 안 오른다"까지는 볼 수 있어도, 그게
히터 소자 열화인지 전원 문제인지는 구분할 계측이 없다 — **exchanger 를 막았던
바로 그 원인(전용 계측 부재)이 heat/cool 에도 그대로 적용된다.**

## 요약

| 이유 | exchanger 와 같은가? |
|---|---|
| ① heat=1-cool 완전 상보 → 릴레이 구분 불가 | heat/cool 만의 고유 문제 (exchanger엔 없음) |
| ② 소수 상태 표본 부족 (I2 heat 0.8%, P3 cool 0.6%) | feeding 의 P4/I2 와 같은 유형 |
| ③ 효과가 너무 커서 다른 신호를 가림 | exchanger 실패의 원인(이유7) 그 자체 |
| ④ 릴레이 자체를 직접 잴 계측이 없음 | exchanger 와 완전히 같은 근본 원인 |


---
# 종합

| 질문 | 결론 |
|---|---|
| 1. wagon 사이클 학습 | 모델을 만든 적이 없다(시각화 도구였음). wagon 주기의 변동계수 4.93, exchanger 는 한 사이클 안에 최대 35회 토글 —애초에 대응이 안 맞는다 |
| 2. 신호 겹침 상세 | heat 는 안 겹친다(버그 수정으로 확인). **feed 가 진짜 범인**(I2 63.2%). feed 를 걷어낸 깨끗한 이벤트도 SNR 0.29~1.06 으로 약하다 |
| 3. 밸브 딜레이 상관관계 | **태그 자체가 없다.** feeding 에만 있는 계측이고 exchanger 엔 없다 |
| 4. heat/cool 센서 | 시도된 적 없음. 구조(완전 상보) + 표본 편중 + 계측 부재가 겹쳐서, 시도해도 exchanger 와 같은 벽에 부딪힌다 |

**공통 결론**: 4가지 질문 모두 최종적으로 같은 지점 — **전용 계측(유량계·밸브
개도·발열량 센서) 부재** — 로 수렴한다. 소프트웨어/통계로 더 밀어붙일 여지가
남아있다기보다, 하드웨어 계측 추가가 유일한 다음 단계로 보인다.